In [1]:
import os, sys
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from pathlib import Path

# ─────────────────────────────────────────────
# 🧭 Set Working Directory
# ─────────────────────────────────────────────

PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Working directory is now:", Path.cwd())

# ─────────────────────────────────────────────
# 🌍 Load Environment
# ─────────────────────────────────────────────

dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")

Working directory is now: C:\Repos\codecritic
✅ Loaded environment variables from env/.env


In [2]:
from app.utilities.github.github_service import validate_token, list_user_repos
import os

# ─────────────────────────────────────────────
# 🔐 Load GitHub credentials from env
# ─────────────────────────────────────────────
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise EnvironmentError("❌ GITHUB_TOKEN is not set in the environment.")

# ─────────────────────────────────────────────
# ✅ Test GitHub Token + Fetch Repos
# ─────────────────────────────────────────────

try:
    user_info = validate_token(GITHUB_TOKEN)
    print(f"✅ Authenticated as: {user_info['login']}")

    repos = list_user_repos(GITHUB_TOKEN)
    print(f"📁 Found {len(repos)} repositories:")
    for r in repos:
        print(f" - {r.full_name} ({'Private' if r.private else 'Public'})")

except Exception as e:
    print("❌ Error during GitHub interaction:", str(e))


✅ Authenticated as: benpodraza
📁 Found 15 repositories:
 - benpodraza/assignment-data-repo (Public)
 - benpodraza/AZ400 (Private)
 - benpodraza/benpodraza (Public)
 - benpodraza/codecritic (Public)
 - benpodraza/codecritic_scoring (Private)
 - benpodraza/concept-builder-static-website (Private)
 - benpodraza/conceptbuilder (Public)
 - benpodraza/eShopOnWeb (Private)
 - benpodraza/github_deployment_test (Private)
 - benpodraza/javatest (Private)
 - benpodraza/k8-jupyter (Private)
 - benpodraza/recommender_system (Private)
 - benpodraza/Terraform_AKS_FastAPI_Microservices (Private)
 - benpodraza/_conceptbuilder_tester (Public)
 - quinnavila/ShopTalk (Private)


In [3]:
from app.utilities.github.github_service import list_repo_tree, get_file_content

# Set these manually for testing
REPO = "benpodraza/_conceptbuilder_tester"
BRANCH = "create_test_set"
PATH = ""  # root folder

# ─────────────────────────────────────────────
# 📁 List repo tree at path
# ─────────────────────────────────────────────

try:
    tree_items = list_repo_tree(GITHUB_TOKEN, REPO, BRANCH, PATH)
    print(f"📂 Items in '{REPO}@{BRANCH}/{PATH or '.'}':")
    for item in tree_items:
        print(f" {item.type.upper():<5} {item.path}")

    # Show debug types
    file_item = next((i for i in tree_items if i.type == "file" or i.type == "blob"), None)
    if file_item is None:
        raise Exception("No files found to preview")

    print("\n🔍 Previewing file:", file_item.path)
    file_content = get_file_content(GITHUB_TOKEN, REPO, BRANCH, file_item.path)

    print("\n📄 File Content (first 10 lines):")
    print("\n".join(file_content.splitlines()[:10]))


except Exception as e:
    print("❌ Error during Stage 2:", str(e))


📂 Items in 'benpodraza/_conceptbuilder_tester@create_test_set/.':
 FILE  .gitignore
 FILE  LICENSE
 FILE  README.md
 DIR   test_files

🔍 Previewing file: .gitignore

📄 File Content (first 10 lines):
# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class

# C extensions
*.so

# Distribution / packaging
.Python


In [4]:
from app.utilities.github.github_service import create_branch, commit_file, create_pull_request, get_file_content
import time

REPO = "benpodraza/_conceptbuilder_tester"
BASE_BRANCH = "create_test_set"
NEW_BRANCH = f"test-branch-{int(time.time())}"
FILE_PATH = "README.md"
COMMIT_MSG = "🔧 Test edit to README.md via GitHub utility"
PR_TITLE = "🧪 Test PR from notebook"
PR_BODY = "This pull request was created as part of Stage 3 testing."

# ─────────────────────────────────────────────
# 1️⃣ Create new branch
# ─────────────────────────────────────────────
print(f"🔀 Creating branch {NEW_BRANCH} from {BASE_BRANCH}...")
branch_ref = create_branch(GITHUB_TOKEN, REPO, BASE_BRANCH, NEW_BRANCH)
print("✅ Branch created:", branch_ref)

# ─────────────────────────────────────────────
# 2️⃣ Modify file content
# ─────────────────────────────────────────────
original = get_file_content(GITHUB_TOKEN, REPO, BASE_BRANCH, FILE_PATH)
new_content = original.strip() + "\n\n> This line was added by a test notebook run."

# ─────────────────────────────────────────────
# 3️⃣ Commit the new version to the branch
# ─────────────────────────────────────────────
print(f"💾 Committing changes to {FILE_PATH} on branch {NEW_BRANCH}...")
commit_sha = commit_file(GITHUB_TOKEN, REPO, NEW_BRANCH, FILE_PATH, new_content, COMMIT_MSG)
print("✅ File committed. SHA:", commit_sha)

# ─────────────────────────────────────────────
# 4️⃣ Create Pull Request
# ─────────────────────────────────────────────
print("📬 Creating pull request...")
pr_url = create_pull_request(GITHUB_TOKEN, REPO, PR_TITLE, head=NEW_BRANCH, base=BASE_BRANCH, body=PR_BODY)
print("✅ Pull request created:")
print(pr_url)


🔀 Creating branch test-branch-1750444231 from create_test_set...
✅ Branch created: refs/heads/test-branch-1750444231
💾 Committing changes to README.md on branch test-branch-1750444231...
✅ File committed. SHA: 67bac5a6d6c001e9be0d7d9a5d826bc2887dfd0d
📬 Creating pull request...
✅ Pull request created:
https://github.com/benpodraza/_conceptbuilder_tester/pull/4


In [5]:
from app.utilities.github.github_service import get_pull_request_diff
import re

# 👇 Paste your PR URL here
PR_URL = pr_url  # or replace with a literal string like: "https://github.com/benpodraza/codecritic/pull/42"

# ─────────────────────────────────────────────
# 🔍 Extract PR number and retrieve diff
# ─────────────────────────────────────────────
try:
    pr_number = int(re.search(r"/pull/(\d+)", PR_URL).group(1))
    print(f"🔢 PR Number: {pr_number}")

    diff_text = get_pull_request_diff(GITHUB_TOKEN, REPO, pr_number)
    print("✅ Retrieved unified diff:\n")

    # Print first 30 lines of diff
    print("\n".join(diff_text.splitlines()[:30]))

except Exception as e:
    print("❌ Failed to retrieve PR diff:", str(e))


🔢 PR Number: 4
✅ Retrieved unified diff:

diff --git a/README.md b/README.md
index 152196d..7e8cc2a 100644
--- a/README.md
+++ b/README.md
@@ -1,3 +1,5 @@
 # _conceptbuilder_tester
 
+> This line was added by a test notebook run.
+
 > This line was added by a test notebook run.
\ No newline at end of file


In [6]:
from app.utilities.github.github_service import get_file_history, get_file_at_commit

FILE_PATH = "README.md"
BRANCH = "create_test_set"

# ─────────────────────────────────────────────
# 🕰️ List history of changes for the file
# ─────────────────────────────────────────────

try:
    history = get_file_history(GITHUB_TOKEN, REPO, FILE_PATH, BRANCH)
    print(f"📜 Found {len(history)} versions of '{FILE_PATH}'")

    for i, v in enumerate(history[:5]):  # Show latest 5 commits
        print(f"{i+1:>2}. [{v.sha[:7]}] {v.date} — {v.author}: {v.message}")

    # ─────────────────────────────────────────────
    # 🔍 Load and compare the two most recent versions
    # ─────────────────────────────────────────────
    if len(history) >= 2:
        latest = history[0]
        previous = history[1]

        latest_content = get_file_at_commit(GITHUB_TOKEN, REPO, latest.sha, FILE_PATH)
        previous_content = get_file_at_commit(GITHUB_TOKEN, REPO, previous.sha, FILE_PATH)

        print("\n🆚 Comparing most recent two versions of README.md")
        print("\n🕒 Newest commit content (first 5 lines):")
        print("\n".join(latest_content.splitlines()[:5]))

        print("\n🕒 Previous commit content (first 5 lines):")
        print("\n".join(previous_content.splitlines()[:5]))

    else:
        print("⚠️ Not enough history to compare two versions.")

except Exception as e:
    print("❌ Error in Stage 5:", str(e))


📜 Found 2 versions of 'README.md'
 1. [c8b8ceb] 2025-06-20T18:13:34Z — Ben Podraza: 🔧 Test edit to README.md via GitHub utility
 2. [411a052] 2025-06-20T17:31:04Z — GitHub: Initial commit

🆚 Comparing most recent two versions of README.md

🕒 Newest commit content (first 5 lines):
# _conceptbuilder_tester

> This line was added by a test notebook run.

🕒 Previous commit content (first 5 lines):
# _conceptbuilder_tester
